In [1]:
# Import packages
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import pandas as pd
from sklearn.metrics import root_mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
#from pyearth import Earth
from keras import Sequential
from keras.layers import Dense, Input
import os
import rasterio as rio
from rasterio.mask import mask
from shapely.geometry import box

In [2]:
# Parameter
FEATURES = ['logBLUE', 'logGREEN']
LABEL = 'AGC'

# Log image location
LOG_PATH = 'gs://gee-ramiqcom-bucket/blueCarbon/sample/sample_'

# Sample path
SAMPLE_PATH = 'multitemporal_sample.csv'

# File format
SUFFIX = '.csv'

# Year list
YEARS = [2019, 2020, 2021]

# Months
MONTHS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# REGIONS list
REGIONS = [
	'Banda_Neira', 'Belitung', 'Biak', 'Bunaken', 'Derawan', 'Gorontalo', 'Kapota', 'Labuan_Bajo', 'Lombok_Selatan', 'Mentawai', 'Parang', 'Rote', 'Ujung_Kulon'
]

# Region names
TEST = [
    { "region": 'Labuan_Bajo', "month": [3, 4, 5, 6], "year": [ 2019 ] },
    { "region": 'Parang', "month": [3, 4], "year": [ 2019 ] },
    { "region": 'Lombok_Selatan', "month": [9, 10], "year": [ 2019 ] },
    { "region": 'Rote', "month": [9, 10], "year": [ 2021 ] }
]

In [3]:
# Sample data
data = [
     {
       "AGC": 0,
       "Keterangan": "100% sand Bajo",
       "logBLUE": 7.05,
       "logGREEN": 7.18
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Belitung",
       "logBLUE": 6.63,
       "logGREEN": 6.88
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Biak",
       "logBLUE": 7.32,
       "logGREEN": 7.42
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Bunaken",
       "logBLUE": 7.05,
       "logGREEN": 7.08
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Derawan",
       "logBLUE": 7.36,
       "logGREEN": 7.45
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Gorontalo",
       "logBLUE": 6.72,
       "logGREEN": 7.15
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Kapota",
       "logBLUE": 7.29,
       "logGREEN": 7.45
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Mandalika",
       "logBLUE": 6.92,
       "logGREEN": 7.09
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Mentawai",
       "logBLUE": 6.96,
       "logGREEN": 7.11
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Parang Field/Visual",
       "logBLUE": 6.72,
       "logGREEN": 6.89
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Rote",
       "logBLUE": 7.45,
       "logGREEN": 7.51
     },
     {
       "AGC": 0,
       "Keterangan": "100% sand Panaitan",
       "logBLUE": 7.33,
       "logGREEN": 7.34
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass TWP Padaido",
       "logBLUE": 3.37,
       "logGREEN": 5.50
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Kwandang",
       "logBLUE": 4.98,
       "logGREEN": 5.61
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Siberut",
       "logBLUE": 2.55,
       "logGREEN": 5.25
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Parang Island (from field data)",
       "logBLUE": 4.69,
       "logGREEN": 5.45
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Rote Island (combined)",
       "logBLUE": 4.69,
       "logGREEN": 5.45
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Panaitan Island",
       "logBLUE": 1.76,
       "logGREEN": 4.61
     },
     {
       "AGC": 44.9,
       "Keterangan": "100% seagrass Panaitan Island (combined)",
       "logBLUE": 4.77,
       "logGREEN": 5.02
     }
  ]
data = pd.DataFrame.from_records(data)
print(data.shape)

(19, 4)


In [4]:
# Turn into np array for input and output
input = data[FEATURES]
output = data[LABEL]

In [5]:
# Random Forest Regressor
rf = RandomForestRegressor(50)
rf.fit(input, output)
print(rf.feature_importances_)

# Predict its own
rf_model = rf.predict(input)
r2_rf_model = np.corrcoef(output, rf_model)[0, 1] ** 2
rmse_rf_model = root_mean_squared_error(output, rf_model)
print(f"RF\nR^2={r2_rf_model}\nRMSE={rmse_rf_model}")

# Linear regression
lr = LinearRegression()
lr.fit(input, output)

# Predict its own
lr_model = lr.predict(input)
r2_lr_model = np.corrcoef(output, lr_model)[0, 1] ** 2
rmse_lr_model = root_mean_squared_error(output, lr_model)
print(f"LR\nR^2={r2_lr_model}\nRMSE={rmse_lr_model}")

# XGBoost
# xgboost = xgb.XGBRegressor()
# xgboost.fit(input, output)

# SVM
svm_model = SVR(kernel="poly")
svm_model.fit(input, output)

# Predict its own
svm_model_result = svm_model.predict(input)
r2_svm_model = np.corrcoef(output, svm_model_result)[0, 1] ** 2
rmse_svm_model = root_mean_squared_error(output, svm_model_result)
print(f"SVM\nR^2={r2_svm_model}\nRMSE={rmse_svm_model}")

# # Mars
# mars = Earth()
# mars.fit(input, output)

# # Predict its own
# mars_model = mars.predict(input)
# r2_mars_model = np.corrcoef(output, mars_model)[0, 1] ** 2
# rmse_mars_model = root_mean_squared_error(output, mars_model)
# print(f"MARS\nR^2={r2_mars_model}\nRMSE={rmse_mars_model}")

# Step wise
def stepwise(logGREEN):
    return 155.34 - (21.383 * logGREEN)

# Predict its own
stepwise_model = stepwise(input["logGREEN"])
r2_stepwise_model = np.corrcoef(output, stepwise_model)[0, 1] ** 2
rmse_stepwise_model = root_mean_squared_error(output, stepwise_model)
print(f"Stepwise\nR^2={r2_stepwise_model}\nRMSE={rmse_stepwise_model}")

[0.5 0.5]
RF
R^2=1.0
RMSE=2.1564154148268387e-14
LR
R^2=0.9307487175225587
RMSE=5.6996251397699575
SVM
R^2=0.9346857156450565
RMSE=6.421621754860515
Stepwise
R^2=0.930031540196805
RMSE=5.731480927670669


# Deep Learning Model #

In [ ]:
# Build model
model = Sequential(
    [
        Input((None, 2)),
        Dense(64, activation="relu"),
        Dense(1, activation="relu"),
    ]
)
model.summary()

# Compile model
model.compile(
	optimizer='Adam',
	loss='mse',
	metrics=['MeanSquaredError']
)

# Fit model
model.fit(
	x=input / 7.5,
	y=output / 44.9,
	epochs=200,
	batch_size=1,
)

In [ ]:
# Function to predict deep learning
def deep_learning(logData):
    return model.predict(logData / 7.5, batch_size=4096*10).flatten() * 44.9

# Predict its own
dl_model = deep_learning(input)
r2_dl_model = np.corrcoef(output, dl_model)[0, 1] ** 2
rmse_dl_model = root_mean_squared_error(output, dl_model)
print(f"Deep learning\nR^2={r2_dl_model}\nRMSE={rmse_dl_model}")

# Apply Model to Image #

## List of file ##

In [6]:
# List of files
files = os.listdir("logImage")
files = list(filter(lambda x: x.split(".")[len(x.split('.')) - 1] == "tif", files))

## Apply model to the file ##

In [13]:
# Apply SVM to every image available

# Run for all
for region in REGIONS:
	files_region = list(filter(lambda x: region in x, files))

	bounds = []

	for file in files_region:
			# Load the Geotiff
			location = f'logImage/{file}'

			data = rio.open(location)

			# Read the geotiff metadata
			profile = data.profile
			width = profile['width']
			height = profile['height']
			bands = data.count

			if (file == files_region[0]):
				bounds = [box(*data.bounds)]

			# Read the geotiff as numpy array
			raster = data.read()
			raster_1 = raster[0:1]

			# Set no data as -9999
			raster[np.isnan(raster)] = -9999

			# Get the raster shape to make it into table
			rasterT = raster.T
			shape_t = rasterT.shape
			reshape = rasterT.reshape(-1, bands)

			# Make the table
			df = pd.DataFrame(reshape, columns=['logGREEN', 'logBLUE'])[['logBLUE', 'logGREEN']]

			# Predict the table
			prediction = svm_model.predict(df)

			# Return the table into image
			prediction_reshape = prediction.reshape(shape_t[0], shape_t[1], -1)
			image_prediction = prediction_reshape.T

			# Mask value as no data
			image_prediction[image_prediction < 0] = 0
			image_prediction[raster_1 == -9999] = -9999

			# Set the driver and parameter to save the geotiff
			profile['count'] = 1
			profile['driver'] = 'COG'

			# Save the raster into a file
			raster_write = rio.open(f'svm/AGC_{file}', 'w', **profile)
			raster_write.write(image_prediction)
			raster_write.close()

			# Mask the new raster using the new boundary
			raster_read = rio.open(f'svm/AGC_{file}', 'r')

			try:
				# Mask the new image with 1st image boundsh
				mask_image, affine = mask(raster_read, bounds, crop=True)

				# Close the current image
				raster_read.close()

				# Update the parameter
				profile['height'] = mask_image.shape[1]
				profile['width'] = mask_image.shape[2]
				profile['transform'] = affine

				# Write the new one
				raster_write = rio.open(f'svm_v2/AGC_{file}', 'w', **profile)
				raster_write.write(mask_image)
				raster_write.close()
			except Exception as error:
				print(error)
			finally:
				raster_read.close()

## Calculate the mean and CV ##

In [16]:
# Read svm result
agc_lists = os.listdir(f'svm')
agc_lists = list(filter(lambda x: x.split(".")[len(x.split('.')) - 1] == "tif", agc_lists))

# Calculate mean and cv per region
for region in REGIONS:
		# List of directory per region
		agc_region = list(filter(lambda x: region in x, agc_lists))

		print(region)

		# Parameter for the output
		profile = {}
		shape = []
		agc_data = []
		raster_1 = []

		# Load all the raster for each region
		for file in agc_region:

				# Load the geotiff
				data = rio.open(f'svm/{file}')

				raster = []

				# First image as reference for parmeter
				if (file == agc_region[0]):
						profile = data.profile
						raster = data.read()
						raster_1 = data.read()
						shape = raster.shape
				else:
						# Other image adjust to main image
						raster = data.read()
						shape_this = raster.shape
						shape_1 = abs(shape[1] - shape_this[1])
						shape_2 = abs(shape[2] - shape_this[2])
						raster = np.pad(
								raster,
								((0, 0), (0, shape_1), (0, shape_2)),
								"constant",
								constant_values=-9999,
						)

				# Set masked value as 0
				raster[raster == -9999] = 0

				# Add the image into one array
				agc_data.append(raster)

		# Set the array into numpy array
		agc_data = np.stack(agc_data)

		# Generate the mean value for index 0
		mean = agc_data.mean(0)

		# Set no data
		mean[raster_1 == -9999] = -9999

		# Calculate the stddev
		stdDev = agc_data.std(0)

		# Calculate the CV
		cv = (stdDev / mean) * 100
		cv[raster_1 == -9999] = -9999

		# Set the drive and no data
		profile['driver'] = 'COG'
		profile['nodata'] = -9999

		# Save the geotiff mean
		raster_write = rio.open(f"mean_v2/AGCMean_{region}_v2.tif", "w", **profile)
		raster_write.write(mean)
		raster_write.close()

		# Save geotiff CV
		raster_write = rio.open(f"cv_v2/AGCCV_{region}_v2.tif", "w", **profile)
		raster_write.write(cv)
		raster_write.close()

Banda_Neira
Belitung


C:\Users\ramiq\AppData\Local\Temp\ipykernel_23840\894304486.py:64: RuntimeWarning: invalid value encountered in divide
  cv = (stdDev / mean) * 100


Biak
Bunaken
Derawan
Gorontalo
Kapota
Labuan_Bajo
Lombok_Selatan
Mentawai
Parang
Rote
Ujung_Kulon


# Accuracy assessment #

In [ ]:
# Function to plot
def asses_plot(ref, predict, r2, rmse, name, region, year, month):
    plt.figure(figsize=(5, 5))
    plt.title(f"{name}_{region}_{year}_{month}\nR^2: {r2}\nRMSE: {rmse}")
    sns.scatterplot(x=ref, y=predict)
    plt.plot(
        [min(ref), max(ref)], [min(ref), max(ref)], linestyle="-", color="red"
    )  # 1:1 line
    plt.xlabel("Reference AGB (gram/m2)")
    plt.ylabel("Prediction AGB (gram/m2)")
    plt.title(f"{name}_{region}_{year}_{month}\nR^2: {r2}\nRMSE: {rmse}")
    plt.grid(True)
    plt.savefig(f"AccuracyAssesment/{name}_{region}_{year}_{month}", facecolor="white")

# Combined data
main_arr = []

# Combined assessment
for x in TEST:
    for y in x['year']:
        for z in x['month']:
            region = x['region']
            path = f'{LOG_PATH}{region}_{y}_{z}{SUFFIX}'

            if (region == 'Rote' and z == 9 and y == 2021):
                data = pd.read_excel('Rote.xlsx')
                loaded = True
            elif (region == 'Parang' and z == 4 and y == 2019):
                data = pd.read_excel('Parang.xlsx')
                loaded = True
            else:
                data = pd.read_csv(path)
                loaded = False
                logData = data[['logBLUE', 'logGREEN']]

            ref = data['reference' if loaded else 'ref']
            true_values = ref

            # Deep learning
            dl_result = data['deeplearning'] if loaded else deep_learning(logData)
            dl_r2 = np.corrcoef(ref, dl_result)[0, 1] ** 2
            dl_rmse = root_mean_squared_error(ref, dl_result)
            asses_plot(ref, dl_result, dl_r2, dl_rmse, 'DeepLearning', region, y, z)

            # SVM
            svm_result = data['svm'] if loaded else svm_model.predict(logData)
            svm_r2 = np.corrcoef(ref, svm_result)[0, 1] ** 2
            svm_rmse = root_mean_squared_error(ref, svm_result)
            asses_plot(ref, svm_result, svm_r2, svm_rmse, 'SVM', region, y, z)

            # RF
            rf_result = data['rf'] if loaded else rf.predict(logData)
            rf_r2 = np.corrcoef(ref, rf_result)[0, 1] ** 2
            rf_rmse = root_mean_squared_error(ref, rf_result)
            asses_plot(ref, rf_result, rf_r2, rf_rmse, "RF", region, y, z)

            # Mars
            mars_result = data['mars'] if loaded else mars.predict(logData)
            mars_r2 = np.corrcoef(ref, mars_result)[0, 1] ** 2
            mars_rmse = root_mean_squared_error(ref, mars_result)
            asses_plot(ref, mars_result, mars_r2, mars_rmse, 'MARS', region, y, z)

            # Linear
            linear_result = data['stepwise'] if loaded else lr.predict(logData)
            linear_r2 = np.corrcoef(ref, linear_result)[0, 1] ** 2
            linear_rmse = root_mean_squared_error(ref, linear_result)
            asses_plot(ref, linear_result, linear_r2, linear_rmse, "Linear", region, y, z)

            # Stepwise
            stepwise_result = data['stepwise'] if loaded else stepwise(data['logGREEN'])
            stepwise_r2 = np.corrcoef(ref, stepwise_result)[0, 1] ** 2  #r2_score(ref, dl_result)
            stepwise_rmse = root_mean_squared_error(ref, stepwise_result)
            asses_plot(ref, stepwise_result, stepwise_r2, stepwise_rmse, 'Stepwise', region, y, z)

            # Data length
            data_length = len(true_values)

            # Region array
            arr_region = np.array([ region ] * data_length)
            arr_year = np.array([ y ] * data_length)
            arr_month = np.array([ z ] * data_length)

            # Stack
            stack = np.stack(( arr_region.astype(str), arr_year.astype(int), arr_month.astype(int), ref.astype(float), stepwise_result.astype(float), mars_result.astype(float), svm_result.astype(float), rf_result.astype(float), dl_result.astype(float) ), axis=1)
            main_arr.append(stack)

In [ ]:
# Export to table
main_arr_stack = np.concatenate(main_arr)
main_df = pd.DataFrame(main_arr_stack, columns=['region', 'year', 'month', 'reference', 'stepwise', 'mars', 'svm', 'rf', 'deeplearning'])
main_df

In [ ]:
with pd.ExcelWriter('accuracy_assessment_v4.xlsx', engine='xlsxwriter', engine_kwargs={'options': {'strings_to_numbers': True}}) as writer:
	for x in TEST:
		for y in x.get('year'):
			for z in x.get('month'):
				region = x.get('region')
				data = main_df[(main_df['region'] == region) & (main_df['year'].astype(float) == y) & (main_df['month'].astype(float) == z)]
				print(data)

				# Export to table
				data.to_excel(writer, sheet_name=f'{region}_{y}_{z}', merge_cells=False)

# Accuracy Assessment Bimonthly

In [ ]:
# Function to plot
def asses_plot(ref, predict, r2, rmse, name, region, year, month):
  print(f'{name}_{region}_{year}_{month}\nR^2: {r2}\nRMSE: {rmse}')

  #plt.figure(figsize=(5, 5))
  #sns.scatterplot(x=ref, y=predict)
  #plt.plot([min(ref), max(ref)], [min(ref), max(ref)], linestyle='-', color='red')  # 1:1 line
  #plt.xlabel("Reference")
  #plt.ylabel("Prediction")
  #plt.title("True vs. Predicted AGB (gram/m2)")
  #plt.grid(True)
  #plt.show()

# Combined data
main_arr_bimonthly = []

# Combined assessment
for x in TEST:
    for y in x.get('year'):
        dl_arr = []
        svm_arr = []
        mars_arr = []
        stepwise_arr = []
        rf_arr = []

        for z in x.get('month'):
            region = x.get('region')
            path = f'{LOG_PATH}{region}_{y}_{z}{SUFFIX}'

            data = pd.read_csv(path)
            ref = data['ref'].to_numpy()
            logData = data[['logBLUE', 'logGREEN']].to_numpy()
            true_values = ref

            # Deep learning
            dl_result = deep_learning(logData)
            dl_arr.append(dl_result)
            print(len(dl_result))

            # SVM
            svm_result = svm_model.predict(logData)
            svm_arr.append(svm_result)

            # Mars
            mars_result = mars.predict(logData)
            mars_arr.append(mars_result)

            # Stepwise
            stepwise_result = stepwise(data['logGREEN'])
            stepwise_arr.append(stepwise_result)

            # RF
            rf_result = rf_model.predict(logData)
            rf_arr.append(rf_result)

        dl_result = np.mean(dl_arr, axis=0)
        svm_result = np.mean(svm_arr, axis=0)
        mars_result = np.mean(mars_arr, axis=0)
        stepwise_result = np.mean(stepwise_arr, axis=0)
        rf_result = np.mean(rf_arr, axis=0)

        dl_r2 = r2_score(ref, dl_result)
        dl_rmse = root_mean_squared_error(ref, dl_result)
        # asses_plot(ref, dl_result, dl_r2, dl_rmse, 'DeepLearning', region, y, z)

        svm_r2 = r2_score(ref, svm_result)
        svm_rmse = root_mean_squared_error(ref, svm_result)
        # asses_plot(ref, svm_result, svm_r2, svm_rmse, 'SVM', region, y, z)

        mars_r2 = r2_score(ref, mars_result)
        mars_rmse = root_mean_squared_error(ref, mars_result)
        # asses_plot(ref, mars_result, mars_r2, mars_rmse, 'MARS', region, y, z)

        stepwise_r2 = r2_score(ref, stepwise_result)
        stepwise_rmse = root_mean_squared_error(ref, stepwise_result)
        # asses_plot(ref, stepwise_result, stepwise_r2, stepwise_rmse, 'Stepwise', region, y, z)

        # Data length
        data_length = len(true_values)

        # Region array
        arr_region = [ region ] * data_length
        arr_year = [ y ] * data_length
        arr_month = [ z ] * data_length

        # Stack
        stack = np.stack(( arr_region, arr_year, arr_month, ref, stepwise_result, mars_result, svm_result, dl_result, rf_result ), axis=1)
        main_arr.append(stack)

main_arr_stack_bimonth = np.concatenate(main_arr)
main_df_bimonth = pd.DataFrame(main_arr_stack, columns=['region', 'year', 'month', 'reference', 'stepwise', 'mars', 'svm', 'deeplearning', 'rf' ])
main_df_bimonth.to_excel('accuracy_assessment_v3_bimonth.xlsx', merge_cells=False)
main_df_bimonth

# Apply to sample data time series #

In [ ]:
# Load sample data to see pattern
sample_data = pd.read_csv(SAMPLE_PATH)
sample_data

In [ ]:
# Regress data
input_data = sample_data[['logBLUE', 'logGREEN']].to_numpy()
svm_prediction = svm_model.predict(input_data)
mars_prediction = mars.predict(input_data)
stepwise_prediction = stepwise(sample_data['logGREEN'])
deeplearning_prediction = deep_learning(input_data)
rf_prediction = rf.predict(input_data)

In [ ]:
# Add predicted data back to the datafram
sample_data['SVM'] = svm_prediction.tolist()
sample_data['MARS'] = mars_prediction.tolist()
sample_data['stepwise'] = stepwise_prediction.tolist()
sample_data['deep_learning'] = deeplearning_prediction.tolist()
sample_data['rf'] = rf_prediction.tolist()
sample_data

In [44]:
# Only select some column
sample_prediction = sample_data[['region', 'year', 'month', 'SVM', 'MARS', 'stepwise', 'deep_learning', 'rf']]

# Save data
sample_monthly = sample_prediction.groupby(['region', 'year', 'month']).agg('mean')
sample_monthly.to_excel('prediction_agb_seagrass_monthly_yearly_region_v3.xlsx', merge_cells=False)

In [ ]:
# Set the month to 2 months
for x in range(1, 13, 2):
	sample_prediction.loc[sample_prediction['month'] == x, 'month'] = x + 1
	sample_prediction.loc[sample_prediction['month'] == x + 1, 'month'] = (x + 1) / 2

sample_prediction.sort_values(by=['month'])

In [46]:
# Save data
sample_bimonthly = sample_prediction.groupby(['region', 'year', 'month']).agg('mean')
sample_bimonthly.to_excel('prediction_agb_seagrass_bimonthly_yearly_region_v3.xlsx', merge_cells=False)

In [ ]:
# Visualize per regions
for x in REGIONS:
	sample_region = sample_prediction[sample_prediction['region'] == x]
	sample_mean = sample_region.groupby('month').agg('mean')

	mars = sample_mean['MARS']
	svm = sample_mean['SVM']
	stepwise = sample_mean['stepwise']
	dl = sample_mean['deep_learning']
	rf = sample_mean['rf']

	plt.figure(figsize=(8, 3))
	plt.grid(False)
	plt.plot(svm, label='SVM', color='green')
	plt.plot(mars, label='MARS', color='red')
	plt.plot(stepwise, label='Stepwise', color='blue')
	plt.plot(dl, label='DL', color='orange')
	plt.plot(rf, label='RF', color='purple')
	plt.legend(title='Model')
	plt.xlabel("Month")
	plt.ylabel("AGC")
	plt.title(f"AGC per month {x}")
	plt.grid(True)
	plt.show()